In [1]:
import yaml
import os
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import yaml
import os.path as op
import scanpy as sc
import numpy as np
import pandas as pd


In [2]:

def read_all_results(root_dir, select = ('overlaps_global_top_k_quantile_count.tsv')):
    """
    Reads all YAML files in a tree of directories starting from root_dir.extended
    Args: 
        root_dir (str): The path to the root directory.

    Returns:
        list: A list of dictionaries, where each dictionary represents the
              content of a YAML file.
    """
    all_results = []
    
    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return all_results

    for dirpath, dirnames, filenames in os.walk(root_dir):

        for filename in filenames:
            if filename.endswith(select) and filename in select:

                filepath = os.path.join(dirpath, filename)
                try:
                    overlaps = pd.read_csv(filepath, sep = '\t', index_col = 0)
                    overlaps['net'] = op.basename(op.dirname(filepath))
                    overlaps['data_config'] = filepath.split('/')[-3]

                    all_results.append(overlaps)
                except:
                    continue
    all_results = pd.concat(all_results)

                    
    return all_results

def process_results(all_overlaps):
    all_overlaps['configuration'] = all_overlaps['method']+'_'+all_overlaps['layer']
    all_overlaps['factor'] = all_overlaps['mean_cell_count']/500
    all_overlaps['factor'] = all_overlaps['factor'].apply(lambda x: min(1, x))
    all_overlaps['weighted_overlap'] = all_overlaps['avg_edges_recovered']*(all_overlaps['factor'])

    all_overlaps = all_overlaps[~all_overlaps.avg_edges_recovered.isna()]
    all_overlaps = all_overlaps[~all_overlaps.mean_cell_count.isna()]
    all_overlaps['percentage_overlap'] = all_overlaps['weighted_overlap']/all_overlaps['gold_standard_edges']

    all_overlaps['precision'] = all_overlaps['weighted_overlap']/(all_overlaps['avg_edges_recovered'] + (all_overlaps['max_possible_edges']*all_overlaps['n_top']))
    return all_overlaps

In [3]:
## get SCGENERAI config
all_overlaps = read_all_results('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/', select=('overlaps_global_top_k_tf.tsv'))
all_overlaps.loc[all_overlaps.method == 'scgenerai_config', 'gold_standard_edges'] = all_overlaps[all_overlaps.method == 'scgenerai_config']['gold_standard_edges']/2
all_overlaps = process_results(all_overlaps)
all_overlaps.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures//data/edge_recovery_tf.tsv', sep = '\t', index=False)

/tmp/ipykernel_657864/2485076723.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[44.  44.  44.  ... 58.5 58.5 58.5]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  all_overlaps.loc[all_overlaps.method == 'scgenerai_config', 'gold_standard_edges'] = all_overlaps[all_overlaps.method == 'scgenerai_config']['gold_standard_edges']/2


In [4]:
all_overlaps = read_all_results('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/', select=('overlaps_global_top_k.tsv'))
all_overlaps.loc[all_overlaps.method == 'scgenerai_config', 'gold_standard_edges'] = all_overlaps[all_overlaps.method == 'scgenerai_config']['gold_standard_edges']/2
all_overlaps = process_results(all_overlaps)
scgenerai = all_overlaps[all_overlaps.method == 'scgenerai_config']
all_overlaps.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/edge_recovery.tsv', sep = '\t', index=False)

/tmp/ipykernel_657864/2558935822.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[44. 44. 44. ... 56. 56. 56.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  all_overlaps.loc[all_overlaps.method == 'scgenerai_config', 'gold_standard_edges'] = all_overlaps[all_overlaps.method == 'scgenerai_config']['gold_standard_edges']/2
